In [14]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datetime import datetime
import lightgbm as lgb


In [15]:
transactions_df = pl.read_parquet(r"C:\Users\asus\Documents\HocTap\HK4\CS116 - Python Programming For ML\Pj-selling website\raw_data\transactions-2025-12.parquet")
transactions_df

customer_id,item_id,price,channel,payment,updated_date,location,quantity
i32,str,"decimal[38,4]",str,str,datetime[μs],i32,i32
8861340,"""0151000000003""",1625000.0000,"""In-Store""","""Tiền mặt""",2025-12-12 16:44:47.793,385,1
8680587,"""5074000000001""",139000.0000,"""SPE""","""Tiền mặt""",2025-12-01 10:28:50.830,1063,1
8220170,"""5856000000001""",480000.0000,"""SPE""","""Tiền mặt""",2025-12-23 09:20:27.653,416,1
7496264,"""6768000000002""",316200.0000,"""SPE""","""Tiền mặt""",2025-12-23 09:06:39.980,336,3
9170522,"""6904000000002""",840000.0000,"""SPE""","""Không xác định""",2025-12-23 09:24:55.640,601,1
…,…,…,…,…,…,…,…
9101369,"""2483000000004""",585000.0000,"""In-Store""","""Tiền mặt""",2025-12-08 06:49:12.620,970,1
9570455,"""6429000000002""",202500.0000,"""In-Store""","""Tiền mặt""",2025-12-08 06:48:30.790,1204,1
8693968,"""1512000000004""",284380.0000,"""SPE""","""Tiền mặt""",2025-12-08 18:58:51.597,1263,2


In [16]:
# Gom nhóm theo ngày để tạo chuỗi thời gian
transactions_df = transactions_df.with_columns(
    pl.col("updated_date").dt.date().alias("date")
)

daily_sales = transactions_df.group_by(["date", "location", "item_id"]).agg([
    pl.col("quantity").sum().alias("qty"),
    pl.col("price").mean().alias("avg_price")
])

### Tạo Features

In [17]:

def make_features(data):
    return data.with_columns([
        pl.col("date").dt.weekday().alias("dow"),
        pl.col("date").dt.day().alias("day"),
        (pl.col("date").dt.weekday() > 5).cast(pl.Int32).alias("is_weekend")
    ])

full_df = make_features(daily_sales)

### Chia tập dữ liệu 

In [ ]:
# Chia tập Train (trước 22/12) và Test (từ 22/12)
train_data = full_df.filter(pl.col("date") < datetime(2025, 12, 22).date())
test_data = full_df.filter(pl.col("date") >= datetime(2025, 12, 22).date())

## Tính toán Dự báo (Prediction)

In [19]:


# Chuẩn bị dữ liệu cho LightGBM
features = ["location", "item_id", "avg_price", "dow", "day", "is_weekend"]
target = "qty"

# Chuyển sang định dạng Numpy/Pandas để đưa vào Model (LGBM chưa support trực tiếp Polars)
X_train = train_data.select(features).to_pandas()
y_train = train_data.select(target).to_pandas()
X_test = test_data.select(features).to_pandas()
y_test = test_data.select(target).to_pandas()

# Định nghĩa category để Model hiểu
cat_features = ["location", "item_id"]
for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 4. Huấn luyện mô hình với mục tiêu tối ưu MAE
model = lgb.LGBMRegressor(
    objective='regression_l1', # l1 chính là tối ưu cho MAE
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], 
          eval_metric='mae', callbacks=[lgb.early_stopping(stopping_rounds=50)])

# 5. Dự đoán và Tổng hợp kết quả
test_data = test_data.with_columns(
    pl.Series(name="pred_qty", values=model.predict(X_test))
)

# Gom nhóm kết quả dự đoán cho cả giai đoạn 22/12 -> 31/12
final_prediction = (
    test_data.group_by(["location", "item_id"])
    .agg(pl.col("pred_qty").sum().alias("qty"))
)

# Tính MAE cuối cùng
actual_totals = test_data.group_by(["location", "item_id"]).agg(pl.col("qty").sum().alias("actual_qty"))
comparison = final_prediction.join(actual_totals, on=["location", "item_id"])
mae_score = (comparison["qty"] - comparison["actual_qty"]).abs().mean()

print(f"MAE của mô hình LightGBM: {mae_score:.4f}")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031685 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5141
[LightGBM] [Info] Number of data points in the train set: 2235140, number of used features: 6
[LightGBM] [Info] Start training from score 1.000000
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[940]	valid_0's l1: 0.755247
MAE của mô hình LightGBM: 1.1073


In [20]:

# Làm sạch bảng dự báo (location, item_id, qty)
final_prediction_table = (
    final_prediction
    .select([
        pl.col("location"),
        pl.col("item_id"),
        pl.col("qty").round(0).cast(pl.Int32) # Làm tròn để thực tế hơn cho việc nhập hàng
    ])
    .sort(["location", "qty"], descending=[False, True])
)

# Hiển thị kết quả dự báo 
print(final_prediction_table)

# 3. Tính toán MAE trên toàn bộ sản phẩm và cửa hàng
mae_final = (
    comparison
    .with_columns(
        (pl.col("qty") - pl.col("actual_qty")).abs().alias("abs_error")
    )
    ["abs_error"].mean()
)

print("\n--- ĐÁNH GIÁ MÔ HÌNH ---")
print(f"MAE trên toàn bộ hệ thống: {mae_final:.4f}")

shape: (627_813, 3)
┌──────────┬───────────────┬─────┐
│ location ┆ item_id       ┆ qty │
│ ---      ┆ ---           ┆ --- │
│ i32      ┆ str           ┆ i32 │
╞══════════╪═══════════════╪═════╡
│ 42       ┆ 4690000000001 ┆ 33  │
│ 42       ┆ 7176000000002 ┆ 30  │
│ 42       ┆ 6697000000003 ┆ 28  │
│ 42       ┆ 6751000000002 ┆ 17  │
│ 42       ┆ 7176000000004 ┆ 17  │
│ …        ┆ …             ┆ …   │
│ 1398     ┆ 0028020000001 ┆ 1   │
│ 1398     ┆ 1452000000007 ┆ 1   │
│ 1398     ┆ 5444000000018 ┆ 1   │
│ 1398     ┆ 2110000000018 ┆ 1   │
│ 1398     ┆ 0020120000033 ┆ 1   │
└──────────┴───────────────┴─────┘

--- ĐÁNH GIÁ MÔ HÌNH ---
MAE trên toàn bộ hệ thống: 1.1073
